# 青木まりこ現象｜分析① 書店 vs 図書館の滞在時間

**研究問い**：書店の滞在時間は図書館と違うのか？

- 原資料：松田千春（2003）「情報探索におけるブラウジング行動：図書館と書店における行動観察を基にして」『Library and Information Science』49, pp.1–31（表5–8）
- PDF：https://www.libinformsci.com/10.46895/lis.49.1/data/index.pdf
- 2002年観察、図書館2館・書店2店、それぞれ10人（合計40人）
- 原論文の表5〜8から転記した個票をこのノートに埋め込み、**Colabでそのまま実行可能**にしています。
- **便意データは含まれません。** この比較だけで青木まりこ現象の原因は分かりません。

In [ ]:
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, kruskal

# Colabで日本語フォントが使用できなければグラフラベルのみ英語表示にする
try:
    import japanize_matplotlib
    JP = True
except ImportError:
    JP = False
    print('Japanese font helper is not installed. Charts will use English labels.')


## 1. 元論文の40名のデータを読み込む

データは論文PDFの表5〜8から転記。元論文の滞在時間（時:分:秒）を原値とし、分に変換して分析します。

In [ ]:
raw_csv = '施設,対象者ID,年代,性別,滞在時間_hms,滞在秒,滞在分\n図書館A,1,30代,女性,0:44:15,2655,44.25\n図書館A,2,20代,女性,0:25:40,1540,25.666666666666668\n図書館A,3,60代,男性,0:50:00,3000,50.0\n図書館A,4,30代,男性,0:15:45,945,15.75\n図書館A,5,30代,女性,1:59:02,7142,119.03333333333333\n図書館A,6,40代,女性,1:47:03,6423,107.05\n図書館A,7,20代,女性,1:52:25,6745,112.41666666666667\n図書館A,8,30代,男性,0:04:07,247,4.116666666666666\n図書館A,9,30代,男性,0:02:27,147,2.45\n図書館A,10,20代,女性,0:20:55,1255,20.916666666666668\n図書館B,1,50代,男性,0:12:39,759,12.65\n図書館B,2,50代,女性,0:04:55,295,4.916666666666667\n図書館B,3,10代,女性,0:46:20,2780,46.333333333333336\n図書館B,4,50代,男性,0:06:19,379,6.316666666666666\n図書館B,5,30代,女性,0:02:18,138,2.3\n図書館B,6,20代,女性,0:11:21,681,11.35\n図書館B,7,70代,男性,0:45:35,2735,45.583333333333336\n図書館B,8,20代,女性,0:25:33,1533,25.55\n図書館B,9,20代,男性,0:36:04,2164,36.06666666666667\n図書館B,10,40代,男性,0:06:40,400,6.666666666666667\n書店C,1,40代,女性,0:06:24,384,6.4\n書店C,2,20代,男性,0:20:34,1234,20.566666666666666\n書店C,3,50代,男性,0:20:46,1246,20.766666666666666\n書店C,4,20代,男性,0:10:03,603,10.05\n書店C,5,30代,男性,0:24:51,1491,24.85\n書店C,6,20代,女性,0:25:20,1520,25.333333333333332\n書店C,7,20代,男性,0:20:30,1230,20.5\n書店C,8,20代,男性,0:35:26,2126,35.43333333333333\n書店C,9,30代,男性,0:15:00,900,15.0\n書店C,10,20代,女性,0:24:38,1478,24.633333333333333\n書店D,1,30代,女性,0:27:03,1623,27.05\n書店D,2,20代,男性,0:03:53,233,3.8833333333333333\n書店D,3,40代,女性,0:02:20,140,2.3333333333333335\n書店D,4,30代,女性,0:04:43,283,4.716666666666667\n書店D,5,50代,男性,0:38:20,2300,38.333333333333336\n書店D,6,50代,男性,0:24:45,1485,24.75\n書店D,7,20代,女性,0:27:43,1663,27.716666666666665\n書店D,8,30代,男性,0:03:25,205,3.4166666666666665\n書店D,9,20代,女性,0:13:24,804,13.4\n書店D,10,30代,男性,0:15:34,934,15.566666666666666\n'
df = pd.read_csv(io.StringIO(raw_csv))
df['場所区分'] = df['施設'].apply(lambda v: '図書館' if str(v).startswith('図書館') else '書店')
df['滞在分'] = df['滞在秒'] / 60
assert len(df) == 40
assert df['施設'].value_counts().to_dict() == {'図書館A':10,'図書館B':10,'書店C':10,'書店D':10}
display(df.head())


## 2. 記述統計（図書館 vs 書店、4施設別）

In [ ]:
group_order = ['図書館','書店']
facility_order = ['図書館A','図書館B','書店C','書店D']
summary = (df.groupby('場所区分')['滞在分']
           .agg(n='count', 平均='mean', 中央値='median', 標準偏差='std', 最小='min', 最大='max')
           .reindex(group_order).round(2))
by_facility = (df.groupby('施設')['滞在分']
               .agg(n='count', 平均='mean', 中央値='median', 最小='min', 最大='max')
               .reindex(facility_order).round(2))
print('分類別（分）'); display(summary)
print('施設別（分）'); display(by_facility)


## 3. 箱ひげ図＋個票の表示

平均ではなく、個々人の分布や極端な長時間滞在も確認します。

In [ ]:
labels = group_order if JP else ['Library','Bookstore']
values = [df.loc[df['場所区分']==v,'滞在分'].to_numpy() for v in group_order]
fig, ax = plt.subplots(figsize=(8,5))
ax.boxplot(values, tick_labels=labels, showfliers=False, whis=(0,100))
for i, a in enumerate(values, 1):
    ax.scatter(i + np.random.default_rng(50+i).normal(0,.07,len(a)), a, alpha=.65, s=26)
ax.set_ylabel('滞在時間（分）' if JP else 'Dwell time (minutes)')
ax.set_title('図書館 vs 書店（各20人）' if JP else 'Libraries vs Bookstores (n=20 each)')
ax.grid(axis='y',alpha=.2)
plt.tight_layout()
plt.savefig('01_dwell_groups.png', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
values4 = [df.loc[df['施設']==v,'滞在分'].to_numpy() for v in facility_order]
labels4 = facility_order if JP else ['Library A','Library B','Bookstore C','Bookstore D']
fig, ax = plt.subplots(figsize=(9,5))
ax.boxplot(values4, tick_labels=labels4, showfliers=False, whis=(0,100))
for i, a in enumerate(values4, 1):
    ax.scatter(i + np.random.default_rng(80+i).normal(0,.06,len(a)), a, alpha=.65, s=26)
ax.set_ylabel('滞在時間（分）' if JP else 'Dwell time (minutes)')
ax.set_title('施設別の滞在時間（各10人）' if JP else 'Dwell time by facility (n=10 each)')
ax.grid(axis='y',alpha=.2)
plt.tight_layout()
plt.savefig('02_dwell_facilities.png', dpi=200, bbox_inches='tight')
plt.show()


## 4. 探索的な統計検定

- Mann–Whitney U検定：2分類の分布比較（両側）
- Kruskal–Wallis検定：4施設間の比較（参考）
- Cliff's delta：2分類での大小関係の効果量（図書館−書店）

**重要**：施設は2館・2店だけです。同一施設内の観察者に相関があり得るため、個人40人を独立なサンプルとみなすp値は**探索的な参考値**です。

In [ ]:
library = values[0]
bookstore = values[1]
mw = mannwhitneyu(library, bookstore, alternative='two-sided')
kw = kruskal(*values4)
cliff = np.mean(np.sign(library[:,None] - bookstore[None,:]))
print(f'Mann–Whitney U = {mw.statistic:.1f}, p = {mw.pvalue:.4f}')
print(f'Kruskal–Wallis H = {kw.statistic:.3f}, p = {kw.pvalue:.4f}')
print(f"Cliff's delta（図書館−書店）= {cliff:.3f}")
print(f'平均差（図書館−書店）= {library.mean()-bookstore.mean():.2f}分')
print(f'中央値差（図書館−書店）= {np.median(library)-np.median(bookstore):.2f}分')
print(f'図書館Aの90分超滞在者 = {np.sum(values4[0]>90)}人')


## 5. 結果の解釈・限界

**観察結果**：図書館の平均時間は長い一方、中央値の差は小さい。Mann–Whitney U検定では5%水準で有意差が検出されない。

**限界**：
1. 便意を記録していない（滞在時間と便意を結び付けられない）。
2. 図書館・書店が各2施設で、施設差を十分にコントロールできない。
3. 原論文では**書店Cで1時間前後滞在した客を追跡し切れず数字に反映されていない**と明記（p.12）。
4. 図書館Aに90分超の3名がいるため平均は外れ値に敏感。
5. 有意差なしは**差がない証明ではない**。

次の段階では、ATUS等の公開個票で大規模な滞在時間分布を再分析します。ただしATUSに書店単独の場所コードはなく、一般の小売店カテゴリとの比較にとどまります。

In [ ]:
# 結果の保存（Colabのファイル欄から取得可能）
df.to_csv('matsuda_40_data_checked.csv', index=False, encoding='utf-8-sig')
summary.to_csv('matsuda_group_summary.csv', encoding='utf-8-sig')
by_facility.to_csv('matsuda_facility_summary.csv', encoding='utf-8-sig')
print('保存完了：PNG 2枚、CSV 3枚')
